# Notebook 04 — Clustering & Recommendations
**Purpose:** Segment member base and build the per-household recommendation table

---
## What this notebook delivers
1. Optimal number of clusters (elbow + silhouette evidence)
2. Cluster profile table (who is in each segment?)
3. Segment action table (what to offer each segment)
4. Per-household recommendation export

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

from src.config import get_config
from src.features.preprocessing import run_preprocessing

cfg = get_config()
X, df_hh = run_preprocessing()
print(f'Households: {len(df_hh)}')

## 1. Run Clustering Pipeline

In [ ]:
from src.models.clustering import run_clustering

labels, profile, action_table = run_clustering(df_hh, cfg=cfg)

## 2. Cluster Profile Table

In [ ]:
print('Cluster Profile (mean feature values per cluster):')
print(profile.to_string())

## 3. Segment Action Table — The Marketing Deliverable

In [ ]:
print('\n' + '═' * 90)
print('  SEGMENT ACTION TABLE — Hand this to Marketing')
print('═' * 90)
print(action_table.to_string(index=False))

# Visualise segment sizes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(action_table)))

axes[0].barh(action_table['cluster_name'], action_table['cluster_size'],
             color=colors, edgecolor='white')
axes[0].set_xlabel('Number of Households', fontsize=11)
axes[0].set_title('Segment Size', fontsize=12, fontweight='bold')

axes[1].barh(action_table['cluster_name'], action_table['avg_propensity'],
             color=colors, edgecolor='white')
axes[1].set_xlabel('Avg. Propensity Score', fontsize=11)
axes[1].set_title('Average Propensity by Segment', fontsize=12, fontweight='bold')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Member Segmentation Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(cfg.paths.report_dir + 'segment_overview.png', bbox_inches='tight')
plt.show()

## 4. Attach Clusters to Households and Export

In [ ]:
import os
from pathlib import Path

# Build per-household recommendation table
cluster_to_prod = action_table.set_index('cluster')['recommended_product'].to_dict()
cluster_to_name = action_table.set_index('cluster')['cluster_name'].to_dict()

rec_df = df_hh.copy()
rec_df['cluster']              = labels
rec_df['segment_name']         = rec_df['cluster'].map(cluster_to_name)
rec_df['recommended_product']  = rec_df['cluster'].map(cluster_to_prod)

# Save
out_path = cfg.paths.recommendations
Path(out_path).parent.mkdir(parents=True, exist_ok=True)
rec_df.to_parquet(out_path)
print(f'Recommendations saved to {out_path}')

# Summary
print('\nRecommendation distribution:')
print(rec_df['recommended_product'].value_counts().to_string())

## 5. Cluster Profile Visualisation

In [ ]:
from IPython.display import Image, display

cluster_plots = [
    cfg.paths.report_dir + 'clustering_elbow_silhouette.png',
    cfg.paths.report_dir + 'cluster_profiles.png',
]
for path in cluster_plots:
    if os.path.exists(path):
        print(f'\n{os.path.basename(path)}')
        display(Image(path))